# Continuous-Time Quantum Walks on Graphs

**Topics covered**

1. Drawing a graph from its adjacency matrix.
2. Building new graphs/matrices from old ones via:
   - the **tensor (Kronecker) product** $A_1 \otimes A_2$,
   - the **Cartesian product** of graphs $G_1 \square G_2$ whose adjacency matrix is
     $$A_{G_1 \square G_2} \;=\; A_1 \otimes I_{n_2} \;+\; I_{n_1} \otimes A_2,$$
   - the **line graph** $L(G)$, whose vertices are the edges of $G$ and where two vertices
     of $L(G)$ are adjacent iff the corresponding edges of $G$ share an endpoint.
3. Simulating a **continuous-time quantum walk (CTQW)** governed by the unitary
   $$U(t) \;=\; \exp(\mathrm{i}\,t\,A),$$
   so that an initial state $|\psi_0\rangle$ evolves to $|\psi(t)\rangle = U(t)|\psi_0\rangle$.
4. Plotting the **probability distribution** $p_v(t) = |\langle v|\psi(t)\rangle|^2$ at time $t$,
   and the **average mixing matrix**
   $$\widehat{M}_T(u,v) \;=\; \frac{1}{T}\int_0^T |\langle v\,|\,e^{\mathrm{i}\,t\,A}\,|u\rangle|^2 \,\mathrm{d}t,$$
   together with its limit $\widehat{M}_\infty$ obtained from the spectral decomposition of $A$.

**Conventions.** Graphs are simple, undirected, and unweighted unless stated otherwise.
The adjacency matrix $A$ is therefore real symmetric, hence $\mathrm{i} A$ is skew-Hermitian and
$U(t) = e^{\mathrm{i} t A}$ is unitary. We use the convention $\hbar = 1$ and the Hamiltonian
$H = A$ (one could equivalently take $H = L$, the graph Laplacian; we briefly discuss this below).

> **Running on Google Colab.** All dependencies (`numpy`, `scipy`, `matplotlib`, `networkx`)
> are pre-installed on Colab. Just open this notebook there and run the cells in order.


## 0. Setup

Imports and a couple of plotting defaults.


In [ ]:
# If you are NOT on Colab and any of these are missing, uncomment to install:
# !pip install numpy scipy matplotlib networkx

import numpy as np
import scipy.linalg as sla
from scipy.linalg import expm
import matplotlib.pyplot as plt
import networkx as nx

# Reproducibility for any random graphs we draw
rng = np.random.default_rng(seed=20260425)

# Nicer matplotlib defaults
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 140,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "font.family": "DejaVu Sans",
})

print("numpy", np.__version__, "| scipy.linalg OK | matplotlib", plt.matplotlib.__version__, "| networkx", nx.__version__)


## 1. Drawing a graph from an adjacency matrix

Given a symmetric $0/1$ matrix $A \in \{0,1\}^{n\times n}$ with zero diagonal,
the corresponding simple graph $G(A) = (V, E)$ has vertex set $V = \{0, 1, \dots, n-1\}$
and edge set $E = \{ \{i,j\} : A_{ij} = 1,\ i < j \}$.

The function below validates the input, builds a `networkx.Graph`, and draws it.


In [ ]:
def graph_from_adjacency(A, directed=False, tol=1e-12):
    """Build a networkx graph from a numpy adjacency matrix.

    Parameters
    ----------
    A : (n, n) array_like
        Adjacency matrix. May be weighted; entries with |A_ij| <= tol are treated as 0.
    directed : bool
        If True, returns a DiGraph. Otherwise A must be (numerically) symmetric and an
        undirected Graph is returned.
    """
    A = np.asarray(A)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError(f"A must be square; got shape {A.shape}")
    n = A.shape[0]
    if not directed and not np.allclose(A, A.T, atol=tol):
        raise ValueError("Undirected graph requested but A is not symmetric.")

    G = nx.DiGraph() if directed else nx.Graph()
    G.add_nodes_from(range(n))
    iterator = ((i, j) for i in range(n) for j in (range(n) if directed else range(i + 1, n)))
    for i, j in iterator:
        w = A[i, j]
        if abs(w) > tol:
            G.add_edge(i, j, weight=float(w))
    return G


def draw_graph(A, title=None, layout="spring", node_color="#4C9AFF",
               with_labels=True, ax=None, seed=0):
    """Draw the graph encoded by adjacency matrix A."""
    G = graph_from_adjacency(A)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))

    layouts = {
        "spring": lambda: nx.spring_layout(G, seed=seed),
        "circular": lambda: nx.circular_layout(G),
        "kamada_kawai": lambda: nx.kamada_kawai_layout(G),
        "spectral": lambda: nx.spectral_layout(G),
    }
    pos = layouts.get(layout, layouts["spring"])()

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#666", width=1.2)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_color,
                           edgecolors="black", linewidths=0.8, node_size=380)
    if with_labels:
        nx.draw_networkx_labels(G, pos, ax=ax, font_size=10)
    ax.set_title(title or f"Graph on {A.shape[0]} vertices")
    ax.set_axis_off()
    return ax, pos


### Example: a small custom graph

Let
$$A \;=\;
\begin{pmatrix}
0 & 1 & 1 & 0 & 0\\
1 & 0 & 1 & 1 & 0\\
1 & 1 & 0 & 0 & 1\\
0 & 1 & 0 & 0 & 1\\
0 & 0 & 1 & 1 & 0
\end{pmatrix}.$$


In [ ]:
A_demo = np.array([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0],
    [1, 1, 0, 0, 1],
    [0, 1, 0, 0, 1],
    [0, 0, 1, 1, 0],
], dtype=float)

draw_graph(A_demo, title="Demo graph $G$", layout="spring", seed=3)
plt.show()


### A few canonical families

For convenience, we wrap a few standard adjacency matrices.


In [ ]:
def adjacency_path(n):
    """Path graph P_n on n vertices."""
    A = np.zeros((n, n))
    for i in range(n - 1):
        A[i, i + 1] = A[i + 1, i] = 1
    return A

def adjacency_cycle(n):
    """Cycle graph C_n on n vertices."""
    A = adjacency_path(n)
    A[0, n - 1] = A[n - 1, 0] = 1
    return A

def adjacency_complete(n):
    """Complete graph K_n."""
    A = np.ones((n, n)) - np.eye(n)
    return A

def adjacency_complete_bipartite(m, n):
    """K_{m,n}."""
    A = np.zeros((m + n, m + n))
    A[:m, m:] = 1
    A[m:, :m] = 1
    return A

# Quick sanity check by drawing them
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
draw_graph(adjacency_path(6),               title=r"$P_6$",       layout="kamada_kawai", ax=axes[0])
draw_graph(adjacency_cycle(7),              title=r"$C_7$",       layout="circular",     ax=axes[1])
draw_graph(adjacency_complete(5),           title=r"$K_5$",       layout="circular",     ax=axes[2])
draw_graph(adjacency_complete_bipartite(3,4),title=r"$K_{3,4}$",  layout="kamada_kawai", ax=axes[3])
plt.tight_layout(); plt.show()


## 2. New matrices from old: products and line graphs

### 2.1 Tensor (Kronecker) product

For $A_1 \in \mathbb{R}^{n_1\times n_1}$ and $A_2 \in \mathbb{R}^{n_2\times n_2}$, the Kronecker product is the $n_1 n_2 \times n_1 n_2$ block matrix
$$
A_1 \otimes A_2 \;=\;
\begin{pmatrix}
(A_1)_{11}A_2 & \cdots & (A_1)_{1n_1}A_2\\
\vdots        & \ddots & \vdots\\
(A_1)_{n_1 1}A_2 & \cdots & (A_1)_{n_1 n_1}A_2
\end{pmatrix}.
$$
As a graph operation, $A_1 \otimes A_2$ is the adjacency matrix of the **tensor product of graphs**
$G_1 \times G_2$ (also called categorical / direct product): the vertex set is $V_1 \times V_2$ and
$(u_1,u_2) \sim (v_1,v_2)$ iff $u_1 \sim_{G_1} v_1$ **and** $u_2 \sim_{G_2} v_2$.

### 2.2 Cartesian product of graphs

The **Cartesian product** $G_1 \square G_2$ has vertex set $V_1 \times V_2$ and $(u_1,u_2) \sim (v_1,v_2)$ iff
*either* ($u_1 = v_1$ and $u_2 \sim_{G_2} v_2$) *or* ($u_1 \sim_{G_1} v_1$ and $u_2 = v_2$).
Its adjacency matrix is exactly
$$
A_{G_1\square G_2} \;=\; A_1 \otimes I_{n_2} \;+\; I_{n_1} \otimes A_2.
$$
A useful spectral fact: if $A_1$ has eigenpairs $(\lambda_i, u_i)$ and $A_2$ has $(\mu_j, v_j)$,
then $A_{G_1\square G_2}$ has eigenpairs $(\lambda_i + \mu_j,\; u_i \otimes v_j)$.

### 2.3 Line graph

The **line graph** $L(G)$ has the edges of $G$ as its vertices, and two such vertices are adjacent in
$L(G)$ iff the corresponding edges share an endpoint in $G$. If $B$ is the (unsigned) incidence matrix of $G$
(rows = vertices of $G$, columns = edges of $G$, with $B_{ve} = 1$ iff $v$ is an endpoint of $e$), then
$$
A_{L(G)} \;=\; B^\top B \;-\; 2\,I_{|E|}.
$$

The functions below implement all three constructions.


In [ ]:
# ---- 2.1 / 2.2: Kronecker and Cartesian products --------------------------

def kronecker_product(A1, A2):
    """A_1 \otimes A_2 (just a thin wrapper around np.kron)."""
    return np.kron(np.asarray(A1, dtype=float), np.asarray(A2, dtype=float))

def cartesian_product(A1, A2):
    """Adjacency of G_1 \square G_2:  A_1 \otimes I + I \otimes A_2."""
    A1 = np.asarray(A1, dtype=float)
    A2 = np.asarray(A2, dtype=float)
    n1, n2 = A1.shape[0], A2.shape[0]
    return np.kron(A1, np.eye(n2)) + np.kron(np.eye(n1), A2)


# ---- 2.3: Incidence matrix and line graph --------------------------------

def incidence_matrix(A, tol=1e-12):
    """Return (B, edges) where B is the |V| x |E| unsigned incidence matrix
    and `edges` is the list of edges in the same column order as B.

    Self-loops are skipped (line graphs of multigraphs/loops require a
    different convention; we stick to simple graphs)."""
    A = np.asarray(A)
    n = A.shape[0]
    if not np.allclose(A, A.T):
        raise ValueError("Line graph defined here for undirected (symmetric) A.")
    edges = [(i, j) for i in range(n) for j in range(i + 1, n) if abs(A[i, j]) > tol]
    m = len(edges)
    B = np.zeros((n, m))
    for k, (i, j) in enumerate(edges):
        B[i, k] = 1.0
        B[j, k] = 1.0
    return B, edges

def line_graph_adjacency(A):
    """Adjacency matrix of L(G) where G has adjacency A.

    Uses A_{L(G)} = B^T B - 2 I.
    """
    B, edges = incidence_matrix(A)
    L = B.T @ B - 2.0 * np.eye(B.shape[1])
    # Numerical cleanup: edges of L(G) are 0/1
    L[np.abs(L) < 1e-9] = 0.0
    return L, edges


### Demo: products and line graphs

Take $G_1 = P_3$ (a path on 3 vertices) and $G_2 = P_2$ (a single edge). Then:

- $A_{P_3 \times P_2}$  is the **tensor product** adjacency.
- $A_{P_3 \square P_2}$  is the well-known **$3\times2$ grid** (a "ladder" with 3 rungs).
- The line graph of $K_4$ is the **triangular graph** $T(4)$, isomorphic to the octahedron $K_{2,2,2}$.


In [ ]:
A1 = adjacency_path(3)
A2 = adjacency_path(2)

A_tensor    = kronecker_product(A1, A2)
A_cartesian = cartesian_product(A1, A2)

L_K4, edges_K4 = line_graph_adjacency(adjacency_complete(4))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
draw_graph(A1,         title=r"$G_1 = P_3$",                 layout="kamada_kawai", ax=axes[0])
draw_graph(A2,         title=r"$G_2 = P_2$",                 layout="kamada_kawai", ax=axes[1])
draw_graph(A_cartesian,title="$P_3$ □ $P_2$ (ladder)",  layout="kamada_kawai", ax=axes[2], seed=2)
draw_graph(L_K4,       title=r"$L(K_4) \cong K_{2,2,2}$",    layout="circular",     ax=axes[3])
plt.tight_layout(); plt.show()

print("Tensor product P_3 (x) P_2 — adjacency matrix:")
print(A_tensor.astype(int))
print("\nCartesian product P_3 [] P_2 — adjacency matrix:")
print(A_cartesian.astype(int))
print("\nLine graph of K_4 — adjacency matrix:")
print(L_K4.astype(int))


**Spectral sanity check** for the Cartesian product:
the eigenvalues of $A_{G_1 \square G_2}$ should be all sums $\lambda_i + \mu_j$ where
$\lambda_i \in \mathrm{spec}(A_1)$, $\mu_j \in \mathrm{spec}(A_2)$.


In [ ]:
lam1 = np.linalg.eigvalsh(A1)
lam2 = np.linalg.eigvalsh(A2)
expected = np.sort(np.add.outer(lam1, lam2).ravel())
actual   = np.sort(np.linalg.eigvalsh(A_cartesian))
print("Expected eigenvalues (lam_i + mu_j):", np.round(expected, 6))
print("Actual eigenvalues of A_cartesian   :", np.round(actual,   6))
print("Match:", np.allclose(expected, actual))


## 3. Continuous-time quantum walks

### 3.1 Definition

Let $A$ be the adjacency matrix of a graph on vertex set $V$, $|V| = n$. The state space is
$\mathbb{C}^V \cong \mathbb{C}^n$, with computational basis $\{|v\rangle\}_{v \in V}$.
Given an initial state $|\psi_0\rangle \in \mathbb{C}^n$ with $\|\psi_0\| = 1$, the
**continuous-time quantum walk (CTQW)** governed by $A$ evolves it as
$$
|\psi(t)\rangle \;=\; U(t)|\psi_0\rangle,\qquad U(t) := e^{\mathrm{i}\,t\,A}.
$$
Because $A = A^*$ is real symmetric, $U(t)$ is unitary, so $\|\psi(t)\| = 1$ for all $t$.

### 3.2 Two evaluation strategies

We expose two routines:

- **`ctqw_state_expm`** — uses `scipy.linalg.expm` directly. Convenient for small $A$ and
  one-off evaluations of $|\psi(t)\rangle$.
- **`ctqw_state_spectral`** — diagonalises $A = \sum_k \lambda_k\,P_k$ once
  (with $P_k$ = projector onto the $\lambda_k$-eigenspace), then evaluates
  $$
  e^{\mathrm{i}\,t\,A}|\psi_0\rangle \;=\; \sum_k e^{\mathrm{i}\,t\,\lambda_k}\,P_k|\psi_0\rangle
  $$
  for any list of times in $O(n^2)$ per time after a one-off $O(n^3)$ diagonalisation.
  This is the workhorse for time series / animations.

(For graphs in the thousands of vertices one would switch to Krylov methods, e.g.
`scipy.sparse.linalg.expm_multiply`. For the modest sizes in this notebook, dense linear
algebra is plenty.)


In [ ]:
def _validate_state(psi0, n):
    psi0 = np.asarray(psi0, dtype=complex).reshape(-1)
    if psi0.shape[0] != n:
        raise ValueError(f"|psi_0| has dim {psi0.shape[0]}, expected {n}.")
    nrm = np.linalg.norm(psi0)
    if nrm == 0:
        raise ValueError("Zero initial state.")
    return psi0 / nrm


def ctqw_state_expm(A, psi0, t):
    """Evaluate |psi(t)> = exp(i t A) |psi_0> via direct matrix exponential.

    Parameters
    ----------
    A : (n, n) ndarray
        Hermitian (here, real symmetric) Hamiltonian.
    psi0 : (n,) ndarray
        Initial (not necessarily normalised) state. Will be normalised internally.
    t : float
        Evolution time.
    """
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    psi0 = _validate_state(psi0, n)
    U = expm(1j * t * A)
    return U @ psi0


def ctqw_spectral_decompose(A):
    """Eigendecomposition of a real symmetric A. Returns (eigvals, eigvecs)."""
    A = np.asarray(A, dtype=float)
    if not np.allclose(A, A.T, atol=1e-10):
        raise ValueError("A must be symmetric for a CTQW.")
    eigvals, eigvecs = np.linalg.eigh(A)  # eigvecs columns are orthonormal
    return eigvals, eigvecs


def ctqw_state_spectral(eigvals, eigvecs, psi0, times):
    """Evaluate |psi(t)> at multiple times using a precomputed spectral decomposition.

    Returns an array of shape (len(times), n) of complex amplitudes."""
    n = eigvals.shape[0]
    psi0 = _validate_state(psi0, n)
    coeffs = eigvecs.conj().T @ psi0          # shape (n,)  : <phi_k | psi_0>
    times  = np.atleast_1d(np.asarray(times, dtype=float))
    # phases[t_idx, k] = exp(i * t * lambda_k)
    phases = np.exp(1j * np.outer(times, eigvals))
    # |psi(t)> = sum_k coeffs[k] * exp(i t lam_k) * |phi_k>
    return (phases * coeffs) @ eigvecs.T      # shape (T, n)


def ctqw_probabilities(states):
    """Born rule: |amplitude|^2, with shape preserved on the leading axis."""
    return np.abs(states) ** 2


### 3.3 First example: CTQW on the cycle $C_8$

We start the walker localised at vertex $0$, i.e. $|\psi_0\rangle = |0\rangle$, and watch
the probability distribution spread out under $U(t) = e^{\mathrm{i} t A}$.


In [ ]:
n = 8
A_cycle = adjacency_cycle(n)
psi0 = np.zeros(n); psi0[0] = 1.0

eigvals, eigvecs = ctqw_spectral_decompose(A_cycle)

times = np.linspace(0, 6, 200)
states = ctqw_state_spectral(eigvals, eigvecs, psi0, times)
probs  = ctqw_probabilities(states)

# Cross-check expm vs spectral at a single time
t_check = 2.7
p_expm     = ctqw_probabilities(ctqw_state_expm(A_cycle, psi0, t_check))
p_spectral = ctqw_probabilities(ctqw_state_spectral(eigvals, eigvecs, psi0, [t_check])[0])
print(f"max |p_expm - p_spectral| at t={t_check}:  {np.max(np.abs(p_expm - p_spectral)):.2e}")

# Sanity: total probability should stay 1
print("max |sum p(t) - 1| over t:  ", np.max(np.abs(probs.sum(axis=1) - 1)))


## 4. Probability distributions and the average mixing matrix

### 4.1 Snapshot of $p_v(t)$ at a single time


In [ ]:
def plot_probability_snapshot(A, psi0, t, layout="spring", seed=0,
                              title=None, ax=None, cmap="viridis"):
    """Draw G(A) with each vertex coloured/sized by p_v(t)."""
    psi_t = ctqw_state_expm(A, psi0, t)
    p = ctqw_probabilities(psi_t)

    G = graph_from_adjacency(A)
    if ax is None:
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
    pos = {
        "spring":      lambda: nx.spring_layout(G, seed=seed),
        "circular":    lambda: nx.circular_layout(G),
        "kamada_kawai":lambda: nx.kamada_kawai_layout(G),
        "spectral":    lambda: nx.spectral_layout(G),
    }.get(layout, lambda: nx.spring_layout(G, seed=seed))()

    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#888", width=1.0)
    nodes = nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_color=p, cmap=cmap,
        vmin=0, vmax=max(p.max(), 1e-12),
        node_size=200 + 1500 * p,
        edgecolors="black", linewidths=0.6,
    )
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=9)
    ax.set_title(title or fr"CTQW probabilities at $t={t:g}$")
    ax.set_axis_off()
    plt.colorbar(nodes, ax=ax, fraction=0.04, pad=0.02, label=r"$p_v(t)$")
    return ax, p


In [ ]:
# Snapshots at a few times for the cycle C_8
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, t in zip(axes, [0.0, 0.8, 1.6, 3.2]):
    plot_probability_snapshot(A_cycle, psi0, t, layout="circular", ax=ax,
                              title=fr"$C_8$ CTQW, $t={t}$")
plt.tight_layout(); plt.show()


### 4.2 Bar-chart / heat-map view across vertices and time


In [ ]:
def plot_distribution_heatmap(A, psi0, t_max=10.0, n_times=400,
                              title=None, cmap="magma"):
    """Heatmap of p_v(t) on a (vertex, time) grid."""
    eigvals, eigvecs = ctqw_spectral_decompose(A)
    times = np.linspace(0, t_max, n_times)
    states = ctqw_state_spectral(eigvals, eigvecs, psi0, times)
    P = ctqw_probabilities(states)        # (n_times, n)

    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(P.T, aspect="auto", origin="lower",
                   extent=[times[0], times[-1], -0.5, A.shape[0] - 0.5],
                   cmap=cmap)
    ax.set_xlabel("time $t$")
    ax.set_ylabel("vertex $v$")
    ax.set_yticks(range(A.shape[0]))
    ax.set_title(title or r"$p_v(t) = |\langle v|e^{\mathrm{i}tA}|\psi_0\rangle|^2$")
    plt.colorbar(im, ax=ax, label=r"$p_v(t)$")
    plt.tight_layout()
    return ax, times, P

plot_distribution_heatmap(A_cycle, psi0, t_max=12.0, n_times=600,
                          title=r"CTQW on $C_8$ starting at $|0\rangle$")
plt.show()


### 4.3 Comparison: cycle vs path vs complete vs Cartesian product

Different graph structures lead to qualitatively different walks:
- On $C_n$, the wave packet splits, travels in both directions and self-interferes.
- On the path $P_n$, reflections at the boundary produce a distinctive interference pattern.
- On $K_n$, the highly degenerate spectrum yields exact periodic revivals.
- On the grid $P_m \square P_n$, the walk separates into a tensor product of 1D walks.


In [ ]:
examples = [
    ("$C_{12}$",                  adjacency_cycle(12)),
    ("$P_{12}$",                  adjacency_path(12)),
    ("$K_{12}$",                  adjacency_complete(12)),
    ("$P_4$ □ $P_3$ grid", cartesian_product(adjacency_path(4), adjacency_path(3))),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (name, A_ex) in zip(axes.ravel(), examples):
    psi0_ex = np.zeros(A_ex.shape[0]); psi0_ex[0] = 1.0
    eigvals, eigvecs = ctqw_spectral_decompose(A_ex)
    times = np.linspace(0, 8, 400)
    P = ctqw_probabilities(ctqw_state_spectral(eigvals, eigvecs, psi0_ex, times))
    im = ax.imshow(P.T, aspect="auto", origin="lower",
                   extent=[times[0], times[-1], -0.5, A_ex.shape[0] - 0.5],
                   cmap="magma")
    ax.set_title(f"CTQW on {name} from $|0\\rangle$")
    ax.set_xlabel("$t$"); ax.set_ylabel("vertex")
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
plt.tight_layout(); plt.show()


### 4.4 Average mixing matrix

A central object in the theory of CTQWs is the **average (or time-averaged) mixing matrix**
$$
\widehat{M}_T(u, v) \;=\; \frac{1}{T} \int_0^T \big| \big( e^{\mathrm{i}\,t\,A} \big)_{vu} \big|^2 \, \mathrm{d}t.
$$
This is the average probability of measuring the walker at $v$ when started at $u$, over a window $[0,T]$.

**Closed form for the limit.** Decompose $A$ spectrally as
$A = \sum_{r} \lambda_r\,E_r$, where $E_r$ is the orthogonal projector onto the
$\lambda_r$-eigenspace and the sum runs over **distinct** eigenvalues. Then
$$
\widehat{M}_\infty \;:=\; \lim_{T\to\infty} \widehat{M}_T \;=\; \sum_{r} E_r \circ E_r,
$$
where $\circ$ denotes the entrywise (Hadamard) product. Equivalently, on the eigenbasis,
$$
\widehat{M}_\infty(u,v) \;=\; \sum_{r} \big[E_r(u,v)\big]^2.
$$

Below we provide three implementations:

1. **Numerical time-average** $\widehat{M}_T$ via Riemann sums (good for verification, finite $T$).
2. **Exact closed form** $\widehat{M}_\infty$ from the spectral decomposition of $A$.
3. A vectorised version that handles arbitrary multiplicities by grouping eigenvalues.


In [ ]:
def average_mixing_matrix_numeric(A, T, n_steps=2000):
    """Riemann-sum approximation of \widehat{M}_T = (1/T) \int_0^T |U(t)|^{\circ 2} dt.

    Returns an n x n doubly stochastic-ish matrix (rows sum to 1; symmetric here).
    """
    eigvals, eigvecs = ctqw_spectral_decompose(A)
    n = A.shape[0]
    ts = np.linspace(0.0, T, n_steps)
    M = np.zeros((n, n))
    # We could vectorise more, but n_steps loops are cheap and readable
    for t in ts:
        # U(t) = V diag(e^{i t lam}) V^T, since A is real symmetric
        D = np.exp(1j * t * eigvals)
        U = (eigvecs * D) @ eigvecs.T
        M += np.abs(U) ** 2
    return M / n_steps

def _group_by_eigenvalue(eigvals, eigvecs, tol=1e-9):
    """Group columns of eigvecs by (numerically) equal eigenvalues; returns list of projectors."""
    order = np.argsort(eigvals)
    eigvals = eigvals[order]
    eigvecs = eigvecs[:, order]
    projectors = []
    i = 0
    n = len(eigvals)
    while i < n:
        j = i
        while j < n and abs(eigvals[j] - eigvals[i]) < tol:
            j += 1
        V = eigvecs[:, i:j]
        projectors.append((eigvals[i], V @ V.T))
        i = j
    return projectors

def average_mixing_matrix_exact(A, tol=1e-9):
    """Exact \widehat{M}_\infty = \sum_r E_r \circ E_r."""
    eigvals, eigvecs = ctqw_spectral_decompose(A)
    M_inf = np.zeros_like(A, dtype=float)
    for _, E_r in _group_by_eigenvalue(eigvals, eigvecs, tol=tol):
        M_inf += E_r * E_r          # entrywise (Hadamard) square
    return M_inf


**Quick verification.** For $C_8$, $\widehat{M}_T \to \widehat{M}_\infty$ as $T \to \infty$.
We compare the two for moderately large $T$.


In [ ]:
M_inf_C8     = average_mixing_matrix_exact(A_cycle)
M_T_C8       = average_mixing_matrix_numeric(A_cycle, T=400, n_steps=4000)

print("max |M_T - M_inf|:  ", np.max(np.abs(M_T_C8 - M_inf_C8)))
print("Row sums of M_inf (should be 1):", np.round(M_inf_C8.sum(axis=1), 6))
print("M_inf is symmetric:", np.allclose(M_inf_C8, M_inf_C8.T))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, M, name in zip(axes, [M_T_C8, M_inf_C8], ["$\\widehat{M}_{400}$", "$\\widehat{M}_{\\infty}$"]):
    im = ax.imshow(M, cmap="magma", vmin=0, vmax=M_inf_C8.max())
    ax.set_title(name + r" on $C_8$")
    ax.set_xlabel("$v$"); ax.set_ylabel("$u$")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()


### 4.5 A worked example combining everything

We build a **two-factor Cartesian product** $G = C_4 \square P_3$, simulate a CTQW on it,
plot a few snapshots and the limiting average mixing matrix.


In [ ]:
A_a = adjacency_cycle(4)
A_b = adjacency_path(3)
A_grid = cartesian_product(A_a, A_b)
n_grid = A_grid.shape[0]

# Start from corner (vertex 0)
psi0_grid = np.zeros(n_grid); psi0_grid[0] = 1.0

# 1) Show the underlying graph
fig, ax = plt.subplots(figsize=(5.5, 4))
draw_graph(A_grid, title="$C_4$ □ $P_3$", layout="kamada_kawai", ax=ax, seed=1)
plt.show()

# 2) Probability snapshots
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, t in zip(axes, [0.0, 1.0, 2.5, 5.0]):
    plot_probability_snapshot(A_grid, psi0_grid, t, layout="kamada_kawai",
                              ax=ax, seed=1, title=fr"$t={t}$")
plt.tight_layout(); plt.show()

# 3) Heat-map of p_v(t)
plot_distribution_heatmap(A_grid, psi0_grid, t_max=12.0, n_times=600,
                          title=r"CTQW on $C_4$ □ $P_3$ from corner")
plt.show()

# 4) Average mixing matrix
M_inf_grid = average_mixing_matrix_exact(A_grid)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(M_inf_grid, cmap="magma")
ax.set_title(r"$\widehat{M}_\infty$ for $C_4$ □ $P_3$")
ax.set_xlabel(r"$v$"); ax.set_ylabel(r"$u$")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()


## 5. Optional: line-graph CTQW and Hamiltonian variants

### 5.1 CTQW on the line graph $L(G)$

The line graph turns "edge-localised" dynamics in $G$ into "vertex-localised" dynamics in $L(G)$.
Below we run a CTQW on $L(K_4)$ — the octahedron $K_{2,2,2}$ — and look at its average mixing matrix.


In [ ]:
A_K4 = adjacency_complete(4)
L_K4, edges_K4 = line_graph_adjacency(A_K4)
n_L = L_K4.shape[0]

psi0_L = np.zeros(n_L); psi0_L[0] = 1.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
draw_graph(A_K4, title=r"$G = K_4$",                     layout="circular",     ax=axes[0])
draw_graph(L_K4, title=r"$L(K_4) \cong K_{2,2,2}$",      layout="circular",     ax=axes[1])
plot_probability_snapshot(L_K4, psi0_L, t=1.5, layout="circular", ax=axes[2],
                          title=r"$p_v(t=1.5)$ on $L(K_4)$")
plt.tight_layout(); plt.show()

M_inf_LK4 = average_mixing_matrix_exact(L_K4)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(M_inf_LK4, cmap="magma")
ax.set_title(r"$\widehat{M}_\infty$ on $L(K_4)$")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()


### 5.2 Hamiltonian variants: $H = A$ vs $H = L = D - A$

Some references use the **graph Laplacian** $L = D - A$ (or the normalised version
$\mathcal{L} = I - D^{-1/2} A D^{-1/2}$) as the Hamiltonian instead of $A$.
For **regular** graphs, $D = k I$, so $L = kI - A$ and $e^{\mathrm{i} t L} = e^{\mathrm{i} k t}\,e^{-\mathrm{i} t A}$,
which differs from $e^{\mathrm{i} t A}$ only by a global phase plus time-reversal — the **probabilities** are
identical. For **non-regular** graphs the two choices give genuinely different dynamics.

Below we check this on $C_8$ (regular, identical) and on a star $K_{1,5}$ (non-regular, different).


In [ ]:
def laplacian_from_adjacency(A):
    A = np.asarray(A, dtype=float)
    return np.diag(A.sum(axis=1)) - A

def ctqw_probs_with_hamiltonian(H, psi0, times):
    """|psi(t)>= e^{i t H} |psi_0>; same machinery as before but with arbitrary H = H^*."""
    eigvals, eigvecs = np.linalg.eigh(H)
    states = ctqw_state_spectral(eigvals, eigvecs, psi0, times)
    return ctqw_probabilities(states)


# --- Case 1: regular graph (cycle) --------------------------------------------
A_reg  = adjacency_cycle(8)
L_reg  = laplacian_from_adjacency(A_reg)
psi0_r = np.zeros(8); psi0_r[0] = 1.0
ts = np.linspace(0, 5, 200)
P_A = ctqw_probs_with_hamiltonian(A_reg, psi0_r, ts)
P_L = ctqw_probs_with_hamiltonian(L_reg, psi0_r, ts)
print("Regular case (C_8):  max |P_A - P_L| =", np.max(np.abs(P_A - P_L)))

# --- Case 2: non-regular (star K_{1,5}) ---------------------------------------
A_star  = adjacency_complete_bipartite(1, 5)   # K_{1,5}, a star with 5 leaves
L_star  = laplacian_from_adjacency(A_star)
psi0_s  = np.zeros(6); psi0_s[0] = 1.0          # start at the centre
P_A_s = ctqw_probs_with_hamiltonian(A_star, psi0_s, ts)
P_L_s = ctqw_probs_with_hamiltonian(L_star, psi0_s, ts)
print("Non-regular (K_{1,5}): max |P_A - P_L| =", np.max(np.abs(P_A_s - P_L_s)))

# Visualise the difference on the star
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, P, lbl in zip(axes, [P_A_s, P_L_s], [r"$H = A$", r"$H = L = D-A$"]):
    im = ax.imshow(P.T, aspect="auto", origin="lower",
                   extent=[ts[0], ts[-1], -0.5, 5.5], cmap="magma")
    ax.set_title(fr"CTQW on $K_{{1,5}}$ with {lbl}")
    ax.set_xlabel("$t$"); ax.set_ylabel("vertex")
    plt.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
plt.tight_layout(); plt.show()


## 6. Audience-friendly figures: waves on the edges of $G$

**Goal.** Produce two clean PDFs (`initial.pdf`, `mixing.pdf`) that explain a CTQW on
a *line graph* $L(G)$ to an audience that may not have seen the line-graph
construction before.

**Visual idea.** A vertex of $L(G)$ *is* an edge of $G$, so we draw the **original graph**
$G$ and render each edge $e$ as a **standing wave** whose visible amplitude (and colour
intensity) is proportional to the probability of finding the walker on $e$:

$$
p_e^{\text{(initial)}} \;=\; \delta_{e,e_0}, \qquad
p_e^{\text{(mixed)}} \;=\; \widehat{M}_\infty(e_0,\,e),
$$

where $\widehat{M}_\infty$ is the average-mixing matrix of $L(G)$ from §4.4 and $e_0$
is the edge on which the walker is initially localised.

- **`initial.pdf`** — one big wave on $e_0$, every other edge drawn as a faint baseline.
  The visual narrative: *"the wave starts here."*
- **`mixing.pdf`** — small waves on every edge, sized by $\widehat{M}_\infty(e_0,\,e)$.
  The visual narrative: *"after a long time, the wave is spread (unevenly) across all edges."*

The function below renders one such figure. We then save the two PDFs for two example
graphs: $G = K_4$ (so $L(K_4) \cong K_{2,2,2}$, the octahedron) and $G = C_6$ (so $L(C_6) = C_6$).


In [ ]:
def draw_graph_with_edge_waves(
    A, edge_probs, edge_list,
    title=None, ax=None, layout="circular", seed=0,
    n_wavelengths=2.0, max_amp=0.18,
    base_lw=0.7, max_lw=2.6, cmap="magma", node_color="#222",
    n_samples=160, prob_threshold=1e-4, show_baseline=True, vmax=None,
):
    """Draw the graph G(A) with each edge rendered as a sine wave whose amplitude
    and colour encode a probability p_e.

    Parameters
    ----------
    A : (n, n) ndarray
        Adjacency matrix of the *original* graph G (whose edges are the vertices of L(G)).
    edge_probs : (|E|,) array
        Probability assigned to each edge of G.
    edge_list : list of (i, j) tuples
        Edges of G, in the same order as edge_probs.
    n_wavelengths : float
        Number of full sine wavelengths drawn along each edge.
    max_amp : float
        Wave amplitude (in layout units) for an edge with p_e = max(p).
    base_lw, max_lw : float
        Line widths for the lowest and highest probabilities.
    prob_threshold : float
        Edges with p_e < threshold are NOT given a wave (only the gray baseline is drawn).
    vmax : float or None
        Upper limit for the colormap. If None, uses max(edge_probs).
    """
    n = A.shape[0]
    G = nx.Graph()
    G.add_nodes_from(range(n))
    for (i, j) in edge_list:
        G.add_edge(i, j)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))

    layouts = {
        "circular":     lambda: nx.circular_layout(G),
        "spring":       lambda: nx.spring_layout(G, seed=seed),
        "kamada_kawai": lambda: nx.kamada_kawai_layout(G),
        "spectral":     lambda: nx.spectral_layout(G),
    }
    pos = layouts.get(layout, layouts["circular"])()

    edge_probs = np.asarray(edge_probs, dtype=float)
    p_max_for_colour = vmax if vmax is not None else max(edge_probs.max(), 1e-12)
    norm = plt.Normalize(vmin=0.0, vmax=p_max_for_colour)
    cmap_obj = plt.get_cmap(cmap)

    # Faint baseline straight edge: keeps the topology of G readable
    if show_baseline:
        for (i, j) in edge_list:
            x0, y0 = pos[i]; x1, y1 = pos[j]
            ax.plot([x0, x1], [y0, y1], color="#cccccc", lw=0.6, zorder=1)

    # Draw a sinusoidal wave on each significant edge
    for (i, j), p in zip(edge_list, edge_probs):
        if p < prob_threshold:
            continue
        x0, y0 = pos[i]; x1, y1 = pos[j]
        dx, dy = x1 - x0, y1 - y0
        L = np.hypot(dx, dy)
        if L < 1e-12:
            continue
        nx_, ny_ = -dy / L, dx / L                 # unit normal to the edge
        s = np.linspace(0.0, 1.0, n_samples)
        envelope = np.sin(np.pi * s)               # vanishes smoothly at both endpoints
        amp = max_amp * (p / p_max_for_colour)
        wave = amp * envelope * np.sin(2 * np.pi * n_wavelengths * s)
        xs = x0 + dx * s + nx_ * wave
        ys = y0 + dy * s + ny_ * wave

        lw = base_lw + (max_lw - base_lw) * (p / p_max_for_colour)
        color = cmap_obj(norm(p))
        alpha = 0.35 + 0.65 * (p / p_max_for_colour)
        ax.plot(xs, ys, color=color, lw=lw, alpha=alpha, zorder=2,
                solid_capstyle="round")

    # Nodes on top
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_color,
                           node_size=180, edgecolors="white", linewidths=1.0)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=9, font_color="white")

    # Probability colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label(r"edge probability $p_e$")

    if title:
        ax.set_title(title)
    ax.set_axis_off()
    ax.set_aspect("equal")
    return ax, pos


### 6.1 Build `initial.pdf` and `mixing.pdf`

For each example graph $G$ we:
1. Build $L(G)$ via `line_graph_adjacency`.
2. Pick an initial edge $e_0$ (we use the first edge in the canonical list).
3. Compute $\widehat{M}_\infty$ on $L(G)$ and read off the row $\widehat{M}_\infty(e_0, \cdot)$.
4. Render two panels — one for the initial, one for the mixed distribution.

The PDFs are written to the current working directory; on Colab they appear in the file
browser pane and can be downloaded with `files.download(...)`.


In [ ]:
# Examples to render
example_graphs = [
    ("K_4",  adjacency_complete(4)),
    ("C_6",  adjacency_cycle(6)),
]

# Compute (initial probs, mixed probs) for each example
results = []
for name, A_G in example_graphs:
    L_G, edges_G = line_graph_adjacency(A_G)
    e0_idx = 0                                  # first edge is e_0
    M_inf  = average_mixing_matrix_exact(L_G)
    p_init = np.zeros(len(edges_G)); p_init[e0_idx] = 1.0
    p_mix  = M_inf[e0_idx, :]
    results.append((name, A_G, edges_G, e0_idx, p_init, p_mix))
    print(f"{name}: |E|={len(edges_G)}, e_0={edges_G[e0_idx]}, "
          f"M_inf row sum={p_mix.sum():.6f}, max={p_mix.max():.4f}, min={p_mix.min():.4f}")

# ----- initial.pdf -----
fig, axes = plt.subplots(1, len(results), figsize=(6.5 * len(results), 5.5))
if len(results) == 1:
    axes = [axes]
for ax, (name, A_G, edges_G, e0_idx, p_init, p_mix) in zip(axes, results):
    e0 = edges_G[e0_idx]
    draw_graph_with_edge_waves(
        A_G, p_init, edges_G,
        title=fr"Initial state on $L({name})$" + "\n"
              fr"$|\psi_0\rangle = |e_0\rangle$,  $e_0 = {e0}$",
        ax=ax, layout="circular", vmax=1.0,
    )
fig.suptitle("Initial: a single big wave on $e_0$", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("initial.pdf", bbox_inches="tight")
plt.savefig("initial.png", dpi=140, bbox_inches="tight")
plt.show()

# ----- mixing.pdf -----
fig, axes = plt.subplots(1, len(results), figsize=(6.5 * len(results), 5.5))
if len(results) == 1:
    axes = [axes]
for ax, (name, A_G, edges_G, e0_idx, p_init, p_mix) in zip(axes, results):
    e0 = edges_G[e0_idx]
    draw_graph_with_edge_waves(
        A_G, p_mix, edges_G,
        title=fr"Time-averaged limit on $L({name})$" + "\n"
              fr"$p_e = \widehat{{M}}_\infty(e_0,e)$,  $e_0={e0}$",
        ax=ax, layout="circular",
    )
fig.suptitle("Mixed: small waves on every edge, sized by $\\widehat{M}_\\infty$", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("mixing.pdf", bbox_inches="tight")
plt.savefig("mixing.png", dpi=140, bbox_inches="tight")
plt.show()

print("\nWrote initial.pdf, initial.png, mixing.pdf, mixing.png.")


### 6.2 What the figures are telling us

For $G = C_6$, the line graph is again $C_6$ (a 6-cycle), and the limit
$\widehat{M}_\infty(e_0,\,\cdot)$ is exactly
$$
\Bigl(\tfrac{5}{18},\;\tfrac{1}{9},\;\tfrac{1}{9},\;\tfrac{1}{9},\;\tfrac{5}{18},\;\tfrac{1}{9}\Bigr).
$$
The two large entries sit on $e_0$ and on its **antipodal** edge in $C_6$: a clean
quantum signature absent from the classical random walk, where the limit is uniform
$\frac{1}{6}$ on every vertex.

For $G = K_4$, the line graph is the octahedron $K_{2,2,2}$, and
$\widehat{M}_\infty(e_0,\,\cdot)$ has its two big entries on $e_0$ and on the unique
edge of $K_4$ disjoint from $e_0$ (the antipodal pair in the octahedron). Again the
quantum dynamics privilege antipodes, while the classical chain spreads uniformly.

Try changing `example_graphs` above to add your own graphs — e.g. `("Petersen",
nx.to_numpy_array(nx.petersen_graph()))` or any custom adjacency matrix.

### 6.3 Downloading the PDFs from Google Colab

Uncomment the cell below to download the generated PDFs in a Colab session:

```python
# from google.colab import files
# files.download("initial.pdf")
# files.download("mixing.pdf")
```


## 7. Audience figures: line graphs of tensor products $L(G_1 \otimes G_2)$

We now repeat the *initial / mixing* visual story of §6, but for the **line graph of a tensor
product**.

### 7.1 Definitions and a connectivity caveat

For graphs $G_1 = (V_1, E_1)$ and $G_2 = (V_2, E_2)$, the **tensor (categorical) product**
$G_1 \otimes G_2$ has vertex set $V_1 \times V_2$ and adjacency rule
$$
(v_1, v_2) \,\sim_\otimes\, (v_1', v_2') \iff v_1 \sim_1 v_1'  \;\text{and}\; v_2 \sim_2 v_2'.
$$
Its adjacency matrix is exactly $A_1 \otimes A_2$ (Kronecker product), so we can build it with
the `kronecker_product` function from §2. The line graph is then $L(G_1 \otimes G_2)$, computed
with `line_graph_adjacency` as before.

**Weichsel's theorem (1962).** $G_1 \otimes G_2$ is connected if and only if both $G_1$ and $G_2$
are connected and *at least one* of them is non-bipartite. If both are bipartite, $G_1 \otimes G_2$
splits into exactly two connected components.

**Why this matters for the dynamics.** Connected components are invariant subspaces of the
adjacency matrix, hence of $e^{\mathrm{i} t A}$, hence of $\widehat{M}_\infty$. A walker started
on an edge in one component will *never* reach edges in the other — those will appear with
probability exactly $0$ in `mixing.pdf`. So the visual signature of bipartite/bipartite tensor
products is striking: half the edges of the figure have no wave at all.

### 7.2 Three example pairs

| $(G_1, G_2)$ | Bipartite? | $|V_1 \otimes V_2|$ | $|E(G_1 \otimes G_2)|$ | Connected? |
|---|---|---|---|---|
| $K_3 \otimes K_2$ | $K_3$: no, $K_2$: yes  | 6  | 6  | yes |
| $C_4 \otimes K_2$ | both yes               | 8  | 8  | **no — 2 components** |
| $C_5 \otimes K_2$ | $C_5$: no, $K_2$: yes  | 10 | 10 | yes |

We layout the product graph in the obvious "two-row" way (since $G_2 = K_2$ in all three
examples): bottom row is $V_1 \times \{0\}$, top row is $V_1 \times \{1\}$.


In [ ]:
def product_layout_grid(n1, n2, x_scale=2.0, y_scale=1.0):
    """Place vertex (i, j) of V_1 x V_2 at (x_i, y_j) in a rectangular grid.

    Linear index v = i * n2 + j matches np.kron(A1, A2). Returns dict {v: (x, y)}.
    """
    xs = np.linspace(-1, 1, n1) * x_scale if n1 > 1 else np.array([0.0])
    ys = np.linspace(-1, 1, n2) * y_scale if n2 > 1 else np.array([0.0])
    pos = {}
    for i in range(n1):
        for j in range(n2):
            pos[i * n2 + j] = (xs[i], ys[j])
    return pos


def draw_graph_with_edge_waves_xy(
    A, edge_probs, edge_list, pos,
    title=None, ax=None,
    n_wavelengths=2.0, max_amp=0.10,
    base_lw=0.7, max_lw=2.6, cmap="magma", node_color="#222",
    n_samples=160, prob_threshold=1e-4, vmax=None,
    node_size=240, node_labels=None, label_fontsize=7,
):
    """Same edge-wave drawing as in §6, but with a user-supplied position dict `pos`.

    Useful for product graphs where we want a grid layout rather than circular.
    """
    n = A.shape[0]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4))

    edge_probs = np.asarray(edge_probs, dtype=float)
    p_max_for_colour = vmax if vmax is not None else max(edge_probs.max(), 1e-12)
    norm = plt.Normalize(vmin=0.0, vmax=p_max_for_colour)
    cmap_obj = plt.get_cmap(cmap)

    # baseline straight edges
    for (i, j) in edge_list:
        x0, y0 = pos[i]; x1, y1 = pos[j]
        ax.plot([x0, x1], [y0, y1], color="#cccccc", lw=0.6, zorder=1)

    # waves
    for (i, j), p in zip(edge_list, edge_probs):
        if p < prob_threshold:
            continue
        x0, y0 = pos[i]; x1, y1 = pos[j]
        dx, dy = x1 - x0, y1 - y0
        L = np.hypot(dx, dy)
        if L < 1e-12:
            continue
        nx_, ny_ = -dy / L, dx / L
        s = np.linspace(0.0, 1.0, n_samples)
        envelope = np.sin(np.pi * s)
        amp = max_amp * (p / p_max_for_colour)
        wave = amp * envelope * np.sin(2 * np.pi * n_wavelengths * s)
        xs = x0 + dx * s + nx_ * wave
        ys = y0 + dy * s + ny_ * wave
        lw = base_lw + (max_lw - base_lw) * (p / p_max_for_colour)
        color = cmap_obj(norm(p))
        alpha = 0.35 + 0.65 * (p / p_max_for_colour)
        ax.plot(xs, ys, color=color, lw=lw, alpha=alpha, zorder=2,
                solid_capstyle="round")

    # nodes + labels
    for v in range(n):
        x, y = pos[v]
        ax.plot(x, y, "o", color=node_color, markersize=np.sqrt(node_size),
                markeredgecolor="white", markeredgewidth=1.0, zorder=3)
        if node_labels is not None:
            ax.text(x, y, node_labels[v], ha="center", va="center",
                    color="white", fontsize=label_fontsize, zorder=4)

    sm = plt.cm.ScalarMappable(cmap=cmap_obj, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label(r"$p_e$")

    if title:
        ax.set_title(title, fontsize=10)
    ax.set_axis_off()
    ax.set_aspect("equal")
    return ax


In [ ]:
# Three example pairs (G_1, G_2)
A_K2 = np.array([[0., 1.], [1., 0.]])     # adjacency of K_2

tensor_examples = [
    ("K_3", "K_2", adjacency_complete(3), A_K2, 2.0, 1.0),
    ("C_4", "K_2", adjacency_cycle(4),    A_K2, 2.4, 1.0),
    ("C_5", "K_2", adjacency_cycle(5),    A_K2, 2.8, 1.0),
]

# Compute mixing for each
tensor_results = []
for name1, name2, A1, A2, xs, ys in tensor_examples:
    A_T = kronecker_product(A1, A2)            # (Kronecker = tensor for graphs)
    n1, n2 = A1.shape[0], A2.shape[0]
    pos = product_layout_grid(n1, n2, x_scale=xs, y_scale=ys)
    L_T, edges_T = line_graph_adjacency(A_T)
    # Connectivity check, for narration in the title
    G_T = nx.from_numpy_array(A_T)
    n_comp = nx.number_connected_components(G_T)
    # Average mixing on L(G_1 \otimes G_2)
    M_inf  = average_mixing_matrix_exact(L_T)
    e0_idx = 0
    p_init = np.zeros(len(edges_T)); p_init[e0_idx] = 1.0
    p_mix  = M_inf[e0_idx, :]
    labels = [f"({i},{j})" for i in range(n1) for j in range(n2)]
    tensor_results.append({
        "name1": name1, "name2": name2,
        "A_T": A_T, "edges_T": edges_T, "pos": pos, "labels": labels,
        "p_init": p_init, "p_mix": p_mix, "e0_idx": e0_idx, "n_comp": n_comp,
    })
    e0_pair = edges_T[e0_idx]
    print(f"{name1} (x) {name2}: |V|={A_T.shape[0]}, |E|={len(edges_T)}, "
          f"components={n_comp}, e_0={labels[e0_pair[0]]}-{labels[e0_pair[1]]}, "
          f"max p_mix={p_mix.max():.4f}, min={p_mix.min():.4f}")


### 7.3 Render `initial_tensor.pdf`


In [ ]:
fig, axes = plt.subplots(len(tensor_results), 1, figsize=(9, 3.5 * len(tensor_results)))
if len(tensor_results) == 1:
    axes = [axes]
for ax, R in zip(axes, tensor_results):
    e0_pair = R["edges_T"][R["e0_idx"]]
    e0_label = f"({R['labels'][e0_pair[0]]},{R['labels'][e0_pair[1]]})"
    conn_note = "" if R["n_comp"] == 1 else f"  [disconnected: {R['n_comp']} components]"
    draw_graph_with_edge_waves_xy(
        R["A_T"], R["p_init"], R["edges_T"], R["pos"],
        title=fr"Initial state on $L({R['name1']} \otimes {R['name2']})$:  "
              fr"$|\psi_0\rangle = |e_0\rangle,\ e_0 = $" + e0_label + conn_note,
        ax=ax, vmax=1.0, node_labels=R["labels"],
    )
fig.suptitle(r"Initial: a single big wave on $e_0 \in E(G_1 \otimes G_2)$",
             fontsize=12, y=1.005)
plt.tight_layout()
plt.savefig("initial_tensor.pdf", bbox_inches="tight")
plt.savefig("initial_tensor.png", dpi=140, bbox_inches="tight")
plt.show()
print("Wrote initial_tensor.pdf, initial_tensor.png.")


### 7.4 Render `mixing_tensor.pdf`


In [ ]:
fig, axes = plt.subplots(len(tensor_results), 1, figsize=(9, 3.5 * len(tensor_results)))
if len(tensor_results) == 1:
    axes = [axes]
for ax, R in zip(axes, tensor_results):
    e0_pair = R["edges_T"][R["e0_idx"]]
    e0_label = f"({R['labels'][e0_pair[0]]},{R['labels'][e0_pair[1]]})"
    conn_note = "" if R["n_comp"] == 1 else f"  [disconnected: {R['n_comp']} components]"
    draw_graph_with_edge_waves_xy(
        R["A_T"], R["p_mix"], R["edges_T"], R["pos"],
        title=fr"$\widehat{{M}}_\infty(e_0,\cdot)$ on $L({R['name1']} \otimes {R['name2']})$, "
              fr"$e_0 = $" + e0_label + conn_note,
        ax=ax, node_labels=R["labels"],
    )
fig.suptitle(r"Mixed: $\widehat{M}_\infty(e_0,\cdot)$ on the line graph of the tensor product",
             fontsize=12, y=1.005)
plt.tight_layout()
plt.savefig("mixing_tensor.pdf", bbox_inches="tight")
plt.savefig("mixing_tensor.png", dpi=140, bbox_inches="tight")
plt.show()
print("Wrote mixing_tensor.pdf, mixing_tensor.png.")


### 7.5 What the figures tell us

- **$K_3 \otimes K_2$ (top panel).** Both $G_1$ and $G_2$ are connected and $K_3$ is non-bipartite,
  so by Weichsel the product is connected. The line graph $L(K_3 \otimes K_2)$ has 6 vertices and
  $\widehat{M}_\infty(e_0,\cdot)$ takes only **two** distinct values (one large, one small). The
  picture matches: a few bright yellow waves on the "preferred" edges, smaller purple waves on the
  others.

- **$C_4 \otimes K_2$ (middle panel).** Both factors are bipartite, so by Weichsel the tensor
  product splits into **two connected components**, each isomorphic to $C_4$. The walker started
  on one edge stays in one component forever, and the figure reflects this with no waves at all
  on half the drawn edges (only the gray baseline appears). This is a clean visual demonstration
  of how *graph structure controls quantum-walk support*.

- **$C_5 \otimes K_2$ (bottom panel).** $C_5$ is the smallest non-bipartite cycle, so the product
  is connected (10 vertices, 10 edges). The mixing distribution is more uniform than in the $K_3$
  case but still bimodal — the geometry of the $C_5$ "Möbius-style" wraparound shows up as a
  gentle pattern of larger and smaller waves around the two-row layout.

To extend this list, just add another entry to `tensor_examples` — for example
`("P_4", "C_3", adjacency_path(4), adjacency_cycle(3), 2.5, 1.0)` for a 12-vertex non-bipartite
example, or `("K_4", "K_2", adjacency_complete(4), A_K2, 2.0, 1.0)` for an 8-vertex
"3-cube minus a perfect matching" pattern.

### 7.6 Downloading the PDFs from Google Colab

```python
# from google.colab import files
# files.download("initial_tensor.pdf")
# files.download("mixing_tensor.pdf")
```


## 8. Path graphs $P_n$ and a Godsil-style corollary

This section produces audience figures for the line graph $L(P_n) = P_{n-1}$ and
illustrates a closed-form result on the average mixing matrix.

### 8.1 Statement (Godsil 2011, restated)

Let $G = P_n$ be the path on $n$ vertices, so $L(G) = P_{n-1}$ has $n-1$ vertices, the
edges $e_1, \dots, e_{n-1}$ of $P_n$. Let $E_1, \dots, E_{n-1}$ be the spectral
idempotents of $A_{L(P_n)}$ and $T$ the **flip permutation matrix** acting on
$\mathbb{C}^{n-1}$ via $T|u\rangle = |n-u\rangle$ for $u = 1, \dots, n-1$. Then
$$
\widehat{M}_\infty \;=\; \sum_{r=1}^{n-1} E_r \circ E_r \;=\; \frac{1}{2n}\bigl( 2J + I + T \bigr),
$$
where $J$ is the all-ones $(n-1)\times(n-1)$ matrix and $I$ the identity.

**Diagonal entries — return probabilities.**
- For any edge $q$ that is **not** a centre: $\displaystyle \widehat{M}_{qq} = \frac{3}{2n}$.
- If $n$ is **even**, the centre edge $c$ (i.e. $c = n/2$) satisfies $\displaystyle \widehat{M}_{cc} = \frac{2}{n}$.
- If $n$ is **odd**, there is no centre edge and all diagonal entries equal $\frac{3}{2n}$.

The corollary that follows from Cauchy–Schwarz: the total non-uniform mixing time satisfies
$\mathrm{tn}(P_n, w_e) < \tfrac{1}{(n-1)^n}$ for every edge $e$, with the tightest bound at the
centre edge (when $n$ is even).

### 8.2 Numerical verification

We check the closed form $\widehat{M}_\infty = (2J + I + T)/(2n)$ against the spectral
computation for $n = 3, 4, \dots, 10$.


In [ ]:
def godsil_path_formula(n):
    """Closed form  M_inf = (1/(2n)) * (2J + I + T)  for L(P_n) = P_{n-1}."""
    m = n - 1                            # number of vertices in the line graph
    J = np.ones((m, m))
    I = np.eye(m)
    # T |u> = |n-u> for u = 1..n-1   (1-indexed in the corollary).
    # 0-indexed Python: row m-1-u <- column u
    T = np.zeros((m, m))
    for u in range(m):
        T[m - 1 - u, u] = 1.0
    return (2 * J + I + T) / (2 * n)


print(f"{'n':<4} {'max |M_inf - closed form|':<28} {'M_diag (computed)':<45} {'expected pattern':<25}")
print("-" * 110)
for n in range(3, 11):
    A_P = adjacency_path(n)
    L_P, _ = line_graph_adjacency(A_P)
    M_inf       = average_mixing_matrix_exact(L_P)
    M_predicted = godsil_path_formula(n)
    err = np.max(np.abs(M_inf - M_predicted))
    diag = np.round(np.diag(M_inf), 4).tolist()
    if n % 2 == 0:
        pattern = f"3/(2n)={3/(2*n):.4f}; 2/n={2/n:.4f} at center"
    else:
        pattern = f"all 3/(2n)={3/(2*n):.4f}"
    print(f"{n:<4} {err:<28.2e} {str(diag):<45} {pattern:<25}")


### 8.3 Helper: drawing waves on a horizontal path

We use a dedicated helper that lays $P_n$ out horizontally and renders each edge as a
sinusoidal wave whose amplitude scales with both the probability *and* the edge length
(so larger paths don't make the waves visually overcrowded). The centre edge is
labelled $e_c$ in red when $n$ is even.


In [ ]:
def draw_path_with_edge_waves(
    n_path, edge_probs, ax,
    title=None, vmax=None, n_wavelengths=2.0, cmap="magma",
    base_lw=0.7, max_lw=2.6, n_samples=200, prob_threshold=1e-5,
    show_edge_labels=True, highlight_center=True, show_node_labels=True,
    common_xlim=None,
):
    """Draw P_n on a horizontal line; each edge becomes a sinusoidal wave whose
    amplitude encodes a probability `edge_probs[k]` for edge e_{k+1} = (k, k+1).

    Wave amplitude is scaled to the per-edge length so the figure stays readable
    across a wide range of n.
    """
    pos = {i: (i, 0.0) for i in range(n_path)}    # spacing 1 between consecutive vertices
    edges = [(i, i + 1) for i in range(n_path - 1)]
    edge_probs = np.asarray(edge_probs, dtype=float)
    p_max = vmax if vmax is not None else max(edge_probs.max(), 1e-12)
    norm = plt.Normalize(vmin=0, vmax=p_max)
    cmap_obj = plt.get_cmap(cmap)

    edge_length = 1.0          # by construction
    max_amp = 0.30 * edge_length

    n_edges = n_path - 1
    has_center = (n_edges % 2 == 1)             # n_path even <=> odd-many edges
    center_idx = n_edges // 2 if has_center else None

    # Faint baselines so the path is always readable
    for (i, j) in edges:
        x0, y0 = pos[i]; x1, y1 = pos[j]
        ax.plot([x0, x1], [y0, y1], color="#cccccc", lw=0.6, zorder=1)

    # Sinusoidal waves on each edge above threshold
    for k, ((i, j), p) in enumerate(zip(edges, edge_probs)):
        if p < prob_threshold:
            continue
        x0, y0 = pos[i]; x1, y1 = pos[j]
        s = np.linspace(0.0, 1.0, n_samples)
        envelope = np.sin(np.pi * s)
        amp = max_amp * (p / p_max)
        wave_y = amp * envelope * np.sin(2 * np.pi * n_wavelengths * s)
        xs = x0 + (x1 - x0) * s
        ys = y0 + wave_y
        lw = base_lw + (max_lw - base_lw) * (p / p_max)
        color = cmap_obj(norm(p))
        alpha = 0.35 + 0.65 * (p / p_max)
        ax.plot(xs, ys, color=color, lw=lw, alpha=alpha, zorder=2,
                solid_capstyle="round")

    # ALL edge labels (whether or not the wave is drawn)
    if show_edge_labels:
        for k, (i, j) in enumerate(edges):
            mx = (pos[i][0] + pos[j][0]) / 2
            label = "$e_c$" if (highlight_center and k == center_idx) else f"$e_{{{k+1}}}$"
            color = "#b00000" if (highlight_center and k == center_idx) else "#444"
            weight = "bold"   if (highlight_center and k == center_idx) else "normal"
            ax.text(mx, -max_amp * 1.4, label, ha="center", va="top",
                    fontsize=8, color=color, fontweight=weight)

    # Vertices and (optional) node labels
    for v in range(n_path):
        x, y = pos[v]
        ax.plot(x, y, "o", color="#222", markersize=7,
                markeredgecolor="white", markeredgewidth=1.0, zorder=3)
        if show_node_labels:
            ax.text(x, max_amp * 1.4, f"{v}", ha="center", va="bottom",
                    fontsize=7, color="#666")

    if title:
        ax.set_title(title, fontsize=10, loc="center")
    ax.set_axis_off()
    ax.set_aspect("equal")
    pad = 0.3
    if common_xlim is not None:
        ax.set_xlim(-pad, common_xlim + pad)
    else:
        ax.set_xlim(-pad, n_path - 1 + pad)
    ax.set_ylim(-max_amp * 2.2, max_amp * 2.2)
    return ax


def add_shared_colorbar(fig, axes, cmap, vmin, vmax, label):
    """Add one colorbar to the right of a stack of axes."""
    sm = plt.cm.ScalarMappable(cmap=plt.get_cmap(cmap),
                               norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, fraction=0.025, pad=0.02, shrink=0.8)
    cbar.set_label(label)
    return cbar


### 8.4 Compute initial / mixed / return distributions for several $n$

Set `path_n_values` to whatever list of $n$ you want to compare — it works for any mix
of even and odd values, $n \geq 3$. The default below runs $n = 3, 4, 5, 6, 7, 8$.


In [ ]:
path_n_values = [3, 4, 5, 6, 7, 8]            # change this list freely
e0_choice    = "boundary"                       # "boundary" -> first edge; "center" -> centre edge

path_results = []
for n in path_n_values:
    A_P     = adjacency_path(n)
    L_P, _  = line_graph_adjacency(A_P)
    M_inf   = average_mixing_matrix_exact(L_P)
    n_edges = n - 1

    if e0_choice == "center" and n_edges % 2 == 1:
        e0 = n_edges // 2                       # centre edge index (only when n is even)
    else:
        e0 = 0                                  # boundary edge

    p_init    = np.zeros(n_edges); p_init[e0] = 1.0
    p_mix     = M_inf[e0, :]                    # row of M_inf at e_0
    diag      = np.diag(M_inf)                  # return probabilities M_qq = M_inf(e_q, e_q)

    path_results.append({"n": n, "M_inf": M_inf, "e0": e0,
                         "p_init": p_init, "p_mix": p_mix, "diag": diag})
    print(f"n={n:>2}  e_0=e_{e0+1:<2}  "
          f"sum(p_mix)={p_mix.sum():.4f}  "
          f"max(p_mix)={p_mix.max():.4f}  "
          f"diag pattern={ {round(x, 6) for x in diag} }")


### 8.5 Render the three PDFs


In [ ]:
def render_path_panel_stack(results, mode, suptitle, filename, vmax=None):
    """Render a vertical stack of horizontal-path panels for `results`.

    mode: 'initial', 'mixing', or 'return'.
    """
    n_panels = len(results)
    # Figure width scales with the longest path; height is fixed per panel
    max_n = max(r["n"] for r in results)
    fig, axes = plt.subplots(n_panels, 1,
                             figsize=(1.5 + 0.9 * max_n, 1.7 * n_panels),
                             layout="constrained")
    if n_panels == 1:
        axes = [axes]

    # Decide a common colorbar range so panels are comparable
    if vmax is None:
        if mode == "initial":
            vmax_use = 1.0
        else:
            vmax_use = max(max(r["p_mix" if mode == "mixing" else "diag"]) for r in results)
    else:
        vmax_use = vmax

    for ax, R in zip(axes, results):
        n = R["n"]
        if mode == "initial":
            data = R["p_init"]
            tag  = fr"$P_{{{n}}}$:  $|\psi_0\rangle = |e_{{{R['e0']+1}}}\rangle$"
        elif mode == "mixing":
            data = R["p_mix"]
            tag  = (fr"$P_{{{n}}}$:  $\widehat{{M}}_\infty(e_{{{R['e0']+1}}}, \cdot)$"
                    + ("   ($n$ even, centre edge $e_c$ in red)" if (n - 1) % 2 == 1
                       else "   ($n$ odd, no centre edge)"))
        elif mode == "return":
            data = R["diag"]
            extra = ""
            if (n - 1) % 2 == 1:    # centre exists
                extra = (fr"   $\widehat M_{{cc}} = 2/n = {2/n:.4f}$,  "
                         fr"$\widehat M_{{qq}} = 3/(2n) = {3/(2*n):.4f}$")
            else:
                extra = fr"   all $\widehat M_{{qq}} = 3/(2n) = {3/(2*n):.4f}$"
            tag = fr"$P_{{{n}}}$:  return prob.  $q \mapsto \widehat{{M}}_\infty(e_q, e_q)$" + extra
        else:
            raise ValueError(mode)

        draw_path_with_edge_waves(n, data, ax, title=tag, vmax=vmax_use,
                                  common_xlim=max_n - 1)

    add_shared_colorbar(fig, axes, cmap="magma", vmin=0.0, vmax=vmax_use,
                        label=r"$p_e$")
    fig.suptitle(suptitle, fontsize=12)
    plt.savefig(filename + ".pdf", bbox_inches="tight")
    plt.savefig(filename + ".png", dpi=140, bbox_inches="tight")
    plt.show()
    return fig

# ----- 1) Initial state: a single big wave on e_0 -----
render_path_panel_stack(
    path_results, mode="initial",
    suptitle=r"Initial: a single big wave on $e_0$ in $L(P_n) = P_{n-1}$",
    filename="path_initial",
)

# ----- 2) Time-averaged distribution from e_0 -----
render_path_panel_stack(
    path_results, mode="mixing",
    suptitle=r"Mixed: $\widehat{M}_\infty(e_0, \cdot)$ in $L(P_n)$",
    filename="path_mixing",
)

# ----- 3) Return probabilities (the diagonal of M_inf) -----
render_path_panel_stack(
    path_results, mode="return",
    suptitle=r"Return: $\widehat{M}_\infty(e_q, e_q)$ on $L(P_n)$  —  centre edge stands out for even $n$",
    filename="path_return",
)
print("\nWrote path_initial.{pdf,png}, path_mixing.{pdf,png}, path_return.{pdf,png}.")


### 8.6 What the figures tell us

- **`path_initial.pdf`.** Each panel shows a single tall yellow wave on $e_0 = e_1$
  (the boundary edge); all other edges show only the gray baseline. Identical visual
  starting point for every $n$.
- **`path_mixing.pdf`.** The row $\widehat{M}_\infty(e_1, \cdot)$ is symmetric under the
  flip $q \mapsto n-q$, so the boundary edges $e_1$ and $e_{n-1}$ always carry the largest
  (equal) probability — the closed form gives $\widehat{M}_\infty(e_1, e_1) =
  \widehat{M}_\infty(e_1, e_{n-1}) = (2 + 1 + 1)/(2n) \cdot 1 = 2/n$. Wait — let's be careful.
  Reading off $(2J + I + T)/(2n)$ at the $(1, q)$-entry: $J_{1q} = 1$ everywhere,
  $I_{1q} = \delta_{1q}$, $T_{1q} = \delta_{q, n-1}$, so
  $$
  \widehat{M}_\infty(e_1, e_q) = \frac{2 + \delta_{1q} + \delta_{q,n-1}}{2n}
  \;=\; \begin{cases} 3/(2n) & q = 1\ \text{or}\ q = n-1 \\ 1/n & \text{otherwise.} \end{cases}
  $$
  Both verified by inspection of the panels above.
- **`path_return.pdf`.** This is the heart of the corollary: for each $q$, plot
  $\widehat{M}_\infty(e_q, e_q)$. The $(2J+I+T)$ formula at the diagonal gives
  $\widehat{M}_{qq} = (2 + 1 + \delta_{q,\,n-q})/(2n)$, i.e. $3/(2n)$ unless $q = n-q$
  (only possible when $n$ is even and $q = n/2$, the centre edge), where the bonus from
  $T$ promotes it to $4/(2n) = 2/n$. In the figure, even-$n$ panels show one waving
  edge that is visibly taller than its neighbours — the red-labelled $e_c$ — and odd-$n$
  panels show the same height across all edges.

### 8.7 The Cauchy–Schwarz step in the corollary

For any vector $v$ in $\mathbb{R}_{\geq 0}^{n-1}$ with $\sum_i v_i = 1$, Cauchy–Schwarz gives
$$
\Bigl(\sum_i v_i \cdot 1\Bigr)^2 \;\leq\; (n-1) \sum_i v_i^2,
$$
hence $\sum_i v_i^2 \geq 1/(n-1)$. Applied to the row $\widehat{M}_\infty(e_q, \cdot)$
(which sums to 1), this says $\sum_r \widehat{M}_\infty(e_q, e_r)^2 \geq 1/(n-1)$. From
this and the $(2J+I+T)/(2n)$ form one extracts the bound
$\mathrm{tn}(P_n, w_e) < 1/(n-1)^n$ stated in the corollary, with the centre edge giving
the strictly tighter inequality.

### 8.8 Try your own $n$

Change `path_n_values` (e.g. `[4, 6, 8, 10, 12]` for an even-only sweep, or `[3, 5, 7, 9, 11]`
for odd-only) and rerun §§8.4–8.5. Set `e0_choice = "center"` to start the walker at the
centre edge of an even path — the resulting `path_mixing.pdf` is then perfectly symmetric
about the centre, and the closed-form $\widehat{M}_\infty(e_c, e_c) = 2/n$ shows up as
the tallest wave at the very centre of the panel.

### 8.9 Downloading the PDFs from Google Colab

```python
# from google.colab import files
# files.download("path_initial.pdf")
# files.download("path_mixing.pdf")
# files.download("path_return.pdf")
```


## 9. Summary and pointers for further work

We assembled a small but complete toolkit:

| Function                         | Purpose |
|----------------------------------|---------|
| `graph_from_adjacency`, `draw_graph` | Visualise $G(A)$. |
| `kronecker_product`              | $A_1 \otimes A_2$. |
| `cartesian_product`              | $A_1 \otimes I + I \otimes A_2$, i.e. $A_{G_1 \square G_2}$. |
| `incidence_matrix`, `line_graph_adjacency` | $A_{L(G)} = B^\top B - 2I$. |
| `ctqw_state_expm`, `ctqw_state_spectral`   | $\lvert\psi(t)\rangle = e^{\mathrm{i} t A}\lvert\psi_0\rangle$. |
| `plot_probability_snapshot`, `plot_distribution_heatmap` | Visualise $p_v(t)$. |
| `average_mixing_matrix_numeric`, `average_mixing_matrix_exact` | $\widehat{M}_T$ and $\widehat{M}_\infty$. |
| `draw_graph_with_edge_waves` | Audience-friendly figures: edges of $G$ as sine waves whose amplitude encodes a probability. |
| `product_layout_grid`, `draw_graph_with_edge_waves_xy` | Same as above for product graphs, with an explicit grid layout for $V_1 \times V_2$. |
| `draw_path_with_edge_waves`, `godsil_path_formula` | Path-graph specific: horizontal wave figures for $L(P_n) = P_{n-1}$ and the closed form $\widehat{M}_\infty = (2J+I+T)/(2n)$. |

**Natural next steps.**
- **Perfect / pretty-good state transfer.** Detect times $t$ at which $|U(t)_{uv}| = 1$ (perfect)
  or $\sup_t |U(t)_{uv}| = 1$ (pretty-good). Cycles of certain lengths and Cartesian powers of
  $K_2$ are classical examples.
- **Hitting / mixing times.** Define quantum analogues of hitting times and compare with classical
  random-walk mixing.
- **Mixing on Cayley graphs / strongly regular graphs**, where the spectrum has a closed form and
  $\widehat{M}_\infty$ becomes very explicit.
- **Larger graphs.** Switch the dense `expm` for `scipy.sparse.linalg.expm_multiply` and use
  `scipy.sparse` adjacency matrices.
- **Discrete-time quantum walks** (Szegedy / coined walks) on the same graph families, for comparison.

Have fun!
